# TensorBoard Loss Landscape Plugin Demo: Architecture Comparison

This notebook demonstrates how network architecture fundamentally shapes loss landscapes by comparing 4 different neural network designs on the Boston Housing dataset. We'll visualize how architectural choices affect:

1. **Loss landscape topology** - Sharp vs flat minima
2. **Training dynamics** - Optimization difficulty and stability  
3. **Mode connectivity** - Whether different seeds find connected solutions

## Network Architectures Compared:
- **Tiny Network (8→4→1)**: Undercapacity - too small to fit the data
- **Shallow Network (8→32→1)**: Traditional 2-layer with sharp minima
- **Deep Network (8→64→64→32→16→1)**: 5 layers without regularization - chaotic landscapes
- **Normalized Network (8→64→32→1 + BatchNorm)**: Modern architecture with flat minima

## Slicing Analysis Types:
- **Axis-parallel slicing**: Parameter sensitivity at final trained state
- **Linear interpolation**: Training paths and mode connectivity between seeds
- **Random direction**: 2D loss landscape topology around initial and final states

## Educational Insights:
- How network depth/width affects optimization difficulty
- Why BatchNorm revolutionized deep learning (landscape smoothing)
- Mode connectivity differences between architectures
- Training time vs landscape smoothness trade-offs

## Task 1: Data Setup and Architecture Definitions

In [20]:
# Standard libraries
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# PySlice components
from pysclice.slicers import LinearInterpolationSlicer, AxisParallelSlicer, RandomDirectionSlicer
from pysclice.core import ModelWrapper

# TensorBoard plugin components
import sys
sys.path.append('../../tensorboard_plugin')  # Add path to tensorboard plugin
from tensorboard_loss_slicer.summary_v2 import log_slice
import tensorflow as tf

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

Libraries loaded successfully


In [21]:
# Load Boston Housing dataset
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
data = pd.read_csv('./data/housing.csv', header=None, delimiter=r"\s+", names=column_names)

# Remove outliers (MEDV >= 50.0)
data = data[~(data['MEDV'] >= 50.0)]

# Select features with good correlation to target
feature_cols = ['LSTAT', 'INDUS', 'NOX', 'PTRATIO', 'RM', 'TAX', 'DIS', 'AGE']
X = data[feature_cols].values
y = data['MEDV'].values

# Apply log transformation to reduce skewness
y = np.log1p(y)

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
print(f"Features: {X_train.shape[1]}, Target range: {y_train.min():.3f} - {y_train.max():.3f}")

Dataset loaded: 392 train, 98 test samples
Features: 8, Target range: 1.792 - 3.908


In [22]:
# Define the 4 different network architectures for comparison

class TinyNet(nn.Module):
    """Tiny network - too small to capture dataset complexity"""
    def __init__(self, input_size=8):
        super(TinyNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class ShallowNet(nn.Module):
    """Traditional shallow network - sharp minima expected"""
    def __init__(self, input_size=8):
        super(ShallowNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class DeepNet(nn.Module):
    """Deep network without normalization - chaotic landscape expected"""
    def __init__(self, input_size=8):
        super(DeepNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.fc5(x)
        return x

class NormalizedNet(nn.Module):
    """Modern network with BatchNorm - flat minima expected"""
    def __init__(self, input_size=8):
        super(NormalizedNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

# Architecture configuration
INPUT_SIZE = X_train.shape[1]
architectures = {
    'tiny_net': TinyNet,
    'shallow_net': ShallowNet, 
    'deep_net': DeepNet,
    'normalized_net': NormalizedNet
}

# Display parameter counts
print("Network Architecture Comparison:")
print("=" * 50)
for name, model_class in architectures.items():
    temp_model = model_class(INPUT_SIZE)
    total_params = sum(p.numel() for p in temp_model.parameters())
    print(f"{name:15}: {total_params:5} parameters")
    
print(f"\nDataset: {X_train.shape[0]} train samples, {INPUT_SIZE} features")
print(f"Expected landscape characteristics:")
print(f"- tiny_net: High loss everywhere (underfitting)")
print(f"- shallow_net: Sharp valleys, sensitive to perturbations")  
print(f"- deep_net: Multiple local minima, chaotic landscape")
print(f"- normalized_net: Smooth, wide basins (flat minima)")

Network Architecture Comparison:
tiny_net       :    41 parameters
shallow_net    :   321 parameters
deep_net       :  7361 parameters
normalized_net :  2881 parameters

Dataset: 392 train samples, 8 features
Expected landscape characteristics:
- tiny_net: High loss everywhere (underfitting)
- shallow_net: Sharp valleys, sensitive to perturbations
- deep_net: Multiple local minima, chaotic landscape
- normalized_net: Smooth, wide basins (flat minima)


In [23]:
# Training and evaluation functions
def evaluate_model(model, data_loader, criterion):
    """Evaluate model on given dataset"""
    model.eval()
    total_loss = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)
    
    return total_loss / total_samples

def compute_gradient_norm(model):
    """Compute L2 norm of gradients"""
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

def train_epoch(model, train_loader, optimizer, criterion, writer, epoch):
    """Train one epoch and log metrics"""
    model.train()
    total_loss = 0
    total_samples = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Compute gradient norm before optimizer step
        grad_norm = compute_gradient_norm(model)
        
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        total_samples += inputs.size(0)
        
        # Log batch-level metrics
        global_step = epoch * len(train_loader) + batch_idx
        writer.add_scalar('Train/BatchLoss', loss.item(), global_step)
        writer.add_scalar('Train/GradNorm', grad_norm, global_step)
    
    return total_loss / total_samples

print("Training functions defined")

Training functions defined


## Task 2: Training All Architectures with Multiple Seeds

We'll train each architecture with 2 different random seeds to enable mode connectivity analysis. This will give us 8 total trained models for comparison.

In [24]:
# Training configuration
EPOCHS = 50
LEARNING_RATE = 0.01
LOG_BASE_DIR = './tensorboard_logs/architecture_comparison'

def train_architecture(model_class, arch_name, seed, epochs=50, lr=0.01):
    """Train a specific architecture with given seed"""
    
    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Create fresh model and optimizer  
    model = model_class(INPUT_SIZE)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    print(f"Training {arch_name} with seed {seed}...")
    
    # Storage for checkpoints
    checkpoints = {}
    best_val_loss = float('inf')
    best_epoch = 0
    
    # Training loop
    for epoch in range(epochs):
        # Train one epoch
        train_loss = train_epoch(model, train_loader, optimizer, criterion, 
                               SummaryWriter(f"temp_logs"), epoch)
        
        # Evaluate on validation set
        val_loss = evaluate_model(model, test_loader, criterion)
        
        # Track best model (for potential future use)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
        
        # Save initial and final states
        if epoch == 0:
            checkpoints['initial'] = copy.deepcopy(model.state_dict())
        elif epoch == epochs - 1:
            checkpoints['final'] = copy.deepcopy(model.state_dict())
    
    print(f"  Final: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
    
    return model, checkpoints

# Train all architectures with 2 different seeds
seeds = [42, 123]
trained_models = {}

print("Training all architectures with multiple seeds...")
print("=" * 60)

for arch_name, model_class in architectures.items():
    trained_models[arch_name] = {}
    
    for seed in seeds:
        model, checkpoints = train_architecture(model_class, arch_name, seed, EPOCHS, LEARNING_RATE)
        trained_models[arch_name][f'seed_{seed}'] = {
            'model': model,
            'checkpoints': checkpoints
        }

print(f"\nTraining complete! Trained {len(architectures)} architectures with {len(seeds)} seeds each.")
print(f"Total models: {len(architectures) * len(seeds)}")
print("Ready for loss landscape analysis...")

Training all architectures with multiple seeds...
Training tiny_net with seed 42...
  Final: Train Loss = 0.0333, Val Loss = 0.0326
Training tiny_net with seed 123...
  Final: Train Loss = 0.0333, Val Loss = 0.0326
Training tiny_net with seed 123...
  Final: Train Loss = 0.1317, Val Loss = 0.1253
Training shallow_net with seed 42...
  Final: Train Loss = 0.1317, Val Loss = 0.1253
Training shallow_net with seed 42...
  Final: Train Loss = 0.0308, Val Loss = 0.0293
Training shallow_net with seed 123...
  Final: Train Loss = 0.0308, Val Loss = 0.0293
Training shallow_net with seed 123...
  Final: Train Loss = 0.0294, Val Loss = 0.0265
Training deep_net with seed 42...
  Final: Train Loss = 0.0294, Val Loss = 0.0265
Training deep_net with seed 42...
  Final: Train Loss = 0.1319, Val Loss = 0.1256
Training deep_net with seed 123...
  Final: Train Loss = 0.1319, Val Loss = 0.1256
Training deep_net with seed 123...
  Final: Train Loss = 0.0316, Val Loss = 0.0254
Training normalized_net with s

## Task 3: Loss Landscape Analysis by Architecture

Now we'll create separate TensorBoard runs for each architecture, with multiple slicing analyses as tags within each run.

In [ ]:
# Prepare slice analysis data (smaller subset for faster slicing)
slice_size = 100
slice_indices = np.random.choice(len(X_train), slice_size, replace=False)
slice_X = torch.FloatTensor(X_train[slice_indices])
slice_y = torch.FloatTensor(y_train[slice_indices]).unsqueeze(1)

# Create model wrapper for slicing
def create_model_wrapper(model, state_dict):
    """Create ModelWrapper for slicing analysis"""
    temp_model = copy.deepcopy(model)
    temp_model.load_state_dict(state_dict)
    criterion = nn.MSELoss()
    return ModelWrapper(temp_model, criterion, (slice_X, slice_y))

# Analysis configuration
SLICE_SAMPLES = 50  # Resolution for slicing
SLICE_RANGE = 20.0  # Range for parameter exploration

print(f"Slice analysis configuration:")
print(f"- Data subset: {slice_size} samples")
print(f"- Slice resolution: {SLICE_SAMPLES} samples per slice")
print(f"- Parameter exploration range: ±{SLICE_RANGE}")
print(f"- Analysis structure: 4 runs × 6 tags = 24 total visualizations")

Slice analysis configuration:
- Data subset: 100 samples
- Slice resolution: 30 samples per slice
- Parameter exploration range: ±10.0
- Analysis structure: 4 runs × 6 tags = 24 total visualizations


In [26]:
# Main analysis loop: Create separate run for each architecture
print("Starting loss landscape analysis for all architectures...")
print("=" * 70)

for arch_name in architectures.keys():
    print(f"\nAnalyzing {arch_name}...")
    
    # Create run directory for this architecture
    run_name = f"{arch_name}_lr{LEARNING_RATE}"
    run_log_dir = os.path.join(LOG_BASE_DIR, run_name)
    os.makedirs(run_log_dir, exist_ok=True)
    
    # Create TensorFlow writer for this run
    tf_writer = tf.summary.create_file_writer(run_log_dir)
    
    # Get the two trained models for this architecture
    seed1_data = trained_models[arch_name]['seed_42']
    seed2_data = trained_models[arch_name]['seed_123']
    
    # Create model wrappers
    wrapper_seed1_initial = create_model_wrapper(seed1_data['model'], seed1_data['checkpoints']['initial'])
    wrapper_seed1_final = create_model_wrapper(seed1_data['model'], seed1_data['checkpoints']['final'])
    wrapper_seed2_final = create_model_wrapper(seed2_data['model'], seed2_data['checkpoints']['final'])
    
    print(f"  Created run: {run_name}")
    
    # === TAG 1: Linear Interpolation - Initial to Final ===
    print(f"  - init_to_final")
    linear_slicer = LinearInterpolationSlicer(wrapper_seed1_final)
    
    point_initial = wrapper_seed1_initial.get_parameters()
    point_final = wrapper_seed1_final.get_parameters()
    
    linear_data_init_final = linear_slicer.slice(
        start_point=point_initial,
        end_point=point_final,
        n_samples=SLICE_SAMPLES
    )
    
    with tf_writer.as_default():
        log_slice(
            name="init_to_final",
            slice_data=linear_data_init_final,
            step=EPOCHS-1,
            description=f"Training path for {arch_name}"
        )
    
    # === TAG 2: Linear Interpolation - Seed1 to Seed2 (Mode Connectivity) ===
    print(f"  - seed1_to_seed2")
    point_seed1_final = wrapper_seed1_final.get_parameters()
    point_seed2_final = wrapper_seed2_final.get_parameters()
    
    linear_data_seed1_to_seed2 = linear_slicer.slice(
        start_point=point_seed1_final,
        end_point=point_seed2_final,
        n_samples=SLICE_SAMPLES
    )
    
    with tf_writer.as_default():
        log_slice(
            name="seed1_to_seed2",
            slice_data=linear_data_seed1_to_seed2,
            step=EPOCHS-1,
            description=f"Mode connectivity between seeds for {arch_name}"
        )
    
    # === TAG 3 & 4: Random Direction Slicing - Initial State (2 slices) ===
    print(f"  - landscape_initial_1, landscape_initial_2")
    random_slicer_initial = RandomDirectionSlicer(wrapper_seed1_initial)
    
    for slice_num in [1, 2]:
        random_data_initial = random_slicer_initial.slice(
            center_point=None,
            n_samples=SLICE_SAMPLES,
            x_range=(-SLICE_RANGE, SLICE_RANGE),
            y_range=(-SLICE_RANGE, SLICE_RANGE),
            normalize_directions=True,
            ensure_orthogonal=True
        )
        
        with tf_writer.as_default():
            log_slice(
                name=f"landscape_initial_{slice_num}",
                slice_data=random_data_initial,
                step=0,
                description=f"2D landscape around initialization for {arch_name} (slice {slice_num})"
            )
    
    # === TAG 5 & 6: Random Direction Slicing - Final State (2 slices) ===
    print(f"  - landscape_final_1, landscape_final_2")
    random_slicer_final = RandomDirectionSlicer(wrapper_seed1_final)
    
    for slice_num in [1, 2]:
        random_data_final = random_slicer_final.slice(
            center_point=None,
            n_samples=SLICE_SAMPLES,
            x_range=(-SLICE_RANGE, SLICE_RANGE),
            y_range=(-SLICE_RANGE, SLICE_RANGE),
            normalize_directions=True,
            ensure_orthogonal=True
        )
        
        with tf_writer.as_default():
            log_slice(
                name=f"landscape_final_{slice_num}",
                slice_data=random_data_final,
                step=EPOCHS-1,
                description=f"2D landscape around final model for {arch_name} (slice {slice_num})"
            )
    
    # Close the writer for this architecture
    tf_writer.close()
    print(f"  ✓ Completed analysis for {arch_name}")

print("\n" + "=" * 70)
print("LOSS LANDSCAPE ANALYSIS COMPLETE!")
print("=" * 70)

Starting loss landscape analysis for all architectures...

Analyzing tiny_net...
  Created run: tiny_net_lr0.01
  - init_to_final
  - seed1_to_seed2
  - landscape_initial_1, landscape_initial_2
  - landscape_final_1, landscape_final_2
  - landscape_final_1, landscape_final_2
  ✓ Completed analysis for tiny_net

Analyzing shallow_net...
  Created run: shallow_net_lr0.01
  - init_to_final
  - seed1_to_seed2
  - landscape_initial_1, landscape_initial_2
  ✓ Completed analysis for tiny_net

Analyzing shallow_net...
  Created run: shallow_net_lr0.01
  - init_to_final
  - seed1_to_seed2
  - landscape_initial_1, landscape_initial_2
  - landscape_final_1, landscape_final_2
  - landscape_final_1, landscape_final_2
  ✓ Completed analysis for shallow_net

Analyzing deep_net...
  Created run: deep_net_lr0.01
  - init_to_final
  - seed1_to_seed2
  ✓ Completed analysis for shallow_net

Analyzing deep_net...
  Created run: deep_net_lr0.01
  - init_to_final
  - seed1_to_seed2
  - landscape_initial_1, l

In [27]:
# Analysis Summary and TensorBoard Instructions
print(f"\nAnalysis Summary:")
print(f"- Architectures analyzed: {len(architectures)}")
print(f"- Seeds per architecture: {len(seeds)}")
print(f"- Total trained models: {len(architectures) * len(seeds)}")
print(f"- Slice analyses per architecture: 6 tags")
print(f"- Total visualizations: {len(architectures)} runs × 6 tags = {len(architectures) * 6}")

print(f"\nTensorBoard Run Structure:")
for arch_name in architectures.keys():
    run_name = f"{arch_name}_lr{LEARNING_RATE}"
    print(f"{run_name}:")
    print(f"  ├── axis_parallel_final      (parameter sensitivity)")
    print(f"  ├── init_to_final           (training path)")
    print(f"  ├── seed1_to_seed2          (mode connectivity)")
    print(f"  ├── landscape_initial_1     (2D topology at start)")
    print(f"  ├── landscape_initial_2     (2D topology at start)")
    print(f"  ├── landscape_final_1       (2D topology at end)")
    print(f"  └── landscape_final_2       (2D topology at end)")

print(f"\nExpected Insights by Architecture:")
print(f"• tiny_net: High loss everywhere, minimal learning")
print(f"• shallow_net: Sharp valleys, seed barriers")  
print(f"• deep_net: Chaotic landscapes, training instability")
print(f"• normalized_net: Smooth basins, seed connectivity")

print(f"\nTo view results:")
print(f"tensorboard --logdir {LOG_BASE_DIR}")
print(f"\nTensorBoard Usage Tips:")
print(f"- Select multiple runs to compare architectures")
print(f"- Use tag filters to focus on specific analysis types")  
print(f"- Compare seed1_to_seed2 across architectures for mode connectivity")
print(f"- Compare landscape_final slices to see topology differences")


Analysis Summary:
- Architectures analyzed: 4
- Seeds per architecture: 2
- Total trained models: 8
- Slice analyses per architecture: 6 tags
- Total visualizations: 4 runs × 6 tags = 24

TensorBoard Run Structure:
tiny_net_lr0.01:
  ├── axis_parallel_final      (parameter sensitivity)
  ├── init_to_final           (training path)
  ├── seed1_to_seed2          (mode connectivity)
  ├── landscape_initial_1     (2D topology at start)
  ├── landscape_initial_2     (2D topology at start)
  ├── landscape_final_1       (2D topology at end)
  └── landscape_final_2       (2D topology at end)
shallow_net_lr0.01:
  ├── axis_parallel_final      (parameter sensitivity)
  ├── init_to_final           (training path)
  ├── seed1_to_seed2          (mode connectivity)
  ├── landscape_initial_1     (2D topology at start)
  ├── landscape_initial_2     (2D topology at start)
  ├── landscape_final_1       (2D topology at end)
  └── landscape_final_2       (2D topology at end)
deep_net_lr0.01:
  ├── axis_p

## Educational Analysis: Understanding the Results

This section explains how to interpret the loss landscape visualizations and what they reveal about each architecture.

In [28]:
# Educational Guide: Interpreting Loss Landscape Visualizations

print("How to Interpret the Loss Landscape Results:")
print("=" * 60)

interpretations = {
    "tiny_net": {
        "expected": "High loss everywhere, minimal optimization success",
        "axis_parallel": "Parameters have little effect - model can't fit data",
        "init_to_final": "Flat trajectory at high loss level",
        "seed1_to_seed2": "Both seeds fail similarly - easily connected", 
        "landscapes": "Flat, high-loss regions with little structure"
    },
    
    "shallow_net": {
        "expected": "Sharp minima, sensitive to perturbations",
        "axis_parallel": "High sensitivity - small changes cause big loss spikes",
        "init_to_final": "Sharp descent into narrow valley",
        "seed1_to_seed2": "High barrier between solutions (mode separation)",
        "landscapes": "Deep, narrow valleys with steep walls"
    },
    
    "deep_net": {
        "expected": "Chaotic landscape with many local minima",
        "axis_parallel": "Irregular sensitivity patterns across parameters",
        "init_to_final": "Bumpy trajectory through multiple local minima",
        "seed1_to_seed2": "Very high barriers - chaotic connectivity", 
        "landscapes": "Complex topology with many peaks and valleys"
    },
    
    "normalized_net": {
        "expected": "Smooth, wide basins (flat minima)",
        "axis_parallel": "Low sensitivity - robust to parameter changes",
        "init_to_final": "Smooth descent into wide valley",
        "seed1_to_seed2": "Low/no barrier - same mode connectivity",
        "landscapes": "Wide, shallow basins with gentle slopes"
    }
}

for arch_name, insights in interpretations.items():
    print(f"\n{arch_name.upper()}:")
    print(f"  Overall: {insights['expected']}")
    print(f"  • axis_parallel_final: {insights['axis_parallel']}")
    print(f"  • init_to_final: {insights['init_to_final']}")
    print(f"  • seed1_to_seed2: {insights['seed1_to_seed2']}")
    print(f"  • landscape slices: {insights['landscapes']}")

print(f"\nKey Educational Insights:")
print(f"=" * 60)
print(f"1. Architecture drives landscape shape:")
print(f"   - Network depth/width affects optimization difficulty")
print(f"   - BatchNorm dramatically smooths landscapes")
print(f"   - Capacity mismatch (tiny_net) prevents learning")

print(f"\n2. Mode connectivity reveals solution quality:")
print(f"   - Sharp landscapes: different seeds → different modes")
print(f"   - Flat landscapes: different seeds → same mode")
print(f"   - Connected solutions generalize better")

print(f"\n3. Practical implications:")
print(f"   - Sharp minima: sensitive to weight decay, initialization")
print(f"   - Flat minima: robust training, better generalization")
print(f"   - Deep without regularization: unstable, unpredictable")
print(f"   - Modern architectures: reliable, consistent training")

How to Interpret the Loss Landscape Results:

TINY_NET:
  Overall: High loss everywhere, minimal optimization success
  • axis_parallel_final: Parameters have little effect - model can't fit data
  • init_to_final: Flat trajectory at high loss level
  • seed1_to_seed2: Both seeds fail similarly - easily connected
  • landscape slices: Flat, high-loss regions with little structure

SHALLOW_NET:
  Overall: Sharp minima, sensitive to perturbations
  • axis_parallel_final: High sensitivity - small changes cause big loss spikes
  • init_to_final: Sharp descent into narrow valley
  • seed1_to_seed2: High barrier between solutions (mode separation)
  • landscape slices: Deep, narrow valleys with steep walls

DEEP_NET:
  Overall: Chaotic landscape with many local minima
  • axis_parallel_final: Irregular sensitivity patterns across parameters
  • init_to_final: Bumpy trajectory through multiple local minima
  • seed1_to_seed2: Very high barriers - chaotic connectivity
  • landscape slices: Com